In [43]:
from torch.utils.data import Dataset, DataLoader
import torch
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, HDBSCAN
# GaussianMixture
from sklearn.mixture import GaussianMixture

import matplotlib.pyplot as plt
import yfinance as yf
import seaborn as sns
# import
from hmmlearn.hmm import GaussianHMM
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score
import numpy as np

In [44]:
device ="mps"

In [45]:
with open('../../data/complete_features.pkl', 'rb') as f:
    all_data = pickle.load(f)

In [46]:
data = all_data["scaled_featured"].copy()

In [47]:
all_data.keys()

dict_keys(['scaled_featured', 'Nifty_Close', 'Gold_Close', 'SP500_Close', 'USDINR_Close', 'Nifty_Bank_Close', 'Nifty_IT_Close', 'mean', 'std'])

In [48]:
nifty_close = all_data["Nifty_Close"]
gold_close = all_data["Gold_Close"]
nifty_bank_close = all_data["Nifty_Bank_Close"]
usdinr_close = all_data["USDINR_Close"]
snp500_close = all_data["SP500_Close"]

In [49]:
combined_data = pd.DataFrame({
    "Nifty_Close": nifty_close,
    "Gold_Close": gold_close,
    "Nifty_Bank_Close": nifty_bank_close,
    "USDINR_Close": usdinr_close,
    "SP500_Close": snp500_close
})
# percentage change
combined_data = combined_data.pct_change()


In [51]:
# is tomorrow up or down
combined_data["Nifty"] = (combined_data["Nifty_Close"].shift(-1) > 0).astype(int)
combined_data["Gold"] = (combined_data["Gold_Close"].shift(-1) > 0).astype(int)
combined_data["Nifty_Bank"] = (combined_data["Nifty_Bank_Close"].shift(-1) > 0).astype(int)
combined_data["USDINR"] = (combined_data["USDINR_Close"].shift(-1) > 0).astype(int)
combined_data["SP500"] = (combined_data["SP500_Close"].shift(-1) > 0).astype(int)

In [53]:
combined_data = combined_data[["Nifty", "Gold", "Nifty_Bank", "USDINR", "SP500"]]

In [ ]:

nifty_close = pd.DataFrame(nifty_close)
nifty_close["pct_change"] = nifty_close["Close"].pct_change()

In [7]:
nifty_close["is_tomorrow_up"] = (nifty_close["pct_change"].shift(-1) > 0).astype(int)

In [8]:
nifty_close

,Close,pct_change,is_tomorrow_up
Date,,,
2008-07-07,4030.000000,NaN,0
2008-07-08,3988.550049,-0.010285,1
2008-07-09,4157.100098,0.042258,1
2008-07-10,4162.200195,0.001227,0
2008-07-11,4049.000000,-0.027197,0
...,...,...,...
2025-09-15,25069.199219,-0.001784,1
2025-09-16,25239.099609,0.006777,1
2025-09-17,25330.250000,0.003611,1


In [9]:
data

,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,vol_slope_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20
Date,,,,,,,,,,,,
2008-07-07,-1.220376,-1.882225,2.081321,-0.864592,-1.135796,-0.628471,-0.569023,-2.119483,1.416877,0.823997,-2.165333,-0.568676
2008-07-08,-1.211754,-1.882225,2.077736,-0.022097,-1.117078,-1.046961,-0.542321,-2.455523,1.450259,0.981827,-2.206775,-0.513460
2008-07-09,-0.957426,-1.882225,2.289506,1.636228,-1.218247,-0.338846,-0.561373,-1.988903,1.476862,0.785786,-1.960992,-0.565134
2008-07-10,-0.975703,-1.882225,2.287995,-0.006124,-1.013921,-0.474801,-0.596656,-2.401313,1.493754,0.505013,-1.940271,-0.562557
2008-07-11,-1.136144,-1.882225,2.391761,0.804554,-1.050675,-0.396400,-0.475448,-2.987661,1.507027,0.400347,-2.078101,-0.510752
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-15,0.375855,0.654743,-0.795604,0.021503,0.541634,-0.126723,-0.727148,-1.189892,-0.988672,-0.309058,0.314003,0.507783
2025-09-16,0.280399,0.654743,-0.833663,-0.287568,0.045456,-0.238463,-0.708679,-1.276291,-1.000565,-0.327333,0.453848,0.513354
2025-09-17,0.260114,0.654743,-0.836851,-0.019044,0.154682,-0.363408,-0.703390,-1.335486,-1.009414,-0.239281,0.522324,0.511624


In [10]:
data.head()

,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,vol_slope_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20
Date,,,,,,,,,,,,
2008-07-07,-1.220376,-1.882225,2.081321,-0.864592,-1.135796,-0.628471,-0.569023,-2.119483,1.416877,0.823997,-2.165333,-0.568676
2008-07-08,-1.211754,-1.882225,2.077736,-0.022097,-1.117078,-1.046961,-0.542321,-2.455523,1.450259,0.981827,-2.206775,-0.513460
2008-07-09,-0.957426,-1.882225,2.289506,1.636228,-1.218247,-0.338846,-0.561373,-1.988903,1.476862,0.785786,-1.960992,-0.565134
2008-07-10,-0.975703,-1.882225,2.287995,-0.006124,-1.013921,-0.474801,-0.596656,-2.401313,1.493754,0.505013,-1.940271,-0.562557
2008-07-11,-1.136144,-1.882225,2.391761,0.804554,-1.050675,-0.396400,-0.475448,-2.987661,1.507027,0.400347,-2.078101,-0.510752


In [11]:
data.tail()

,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,vol_slope_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20
Date,,,,,,,,,,,,
2025-09-15,0.375855,0.654743,-0.795604,0.021503,0.541634,-0.126723,-0.727148,-1.189892,-0.988672,-0.309058,0.314003,0.507783
2025-09-16,0.280399,0.654743,-0.833663,-0.287568,0.045456,-0.238463,-0.708679,-1.276291,-1.000565,-0.327333,0.453848,0.513354
2025-09-17,0.260114,0.654743,-0.836851,-0.019044,0.154682,-0.363408,-0.703390,-1.335486,-1.009414,-0.239281,0.522324,0.511624
2025-09-18,0.293060,0.654743,-0.832619,0.038097,0.230366,-0.508229,-0.737254,-1.044970,-1.020331,-0.299090,0.591810,0.309460
2025-09-19,0.080198,0.654743,-0.823297,0.077298,0.164701,-0.717435,-0.742174,-1.146226,-1.028834,-0.229313,0.500421,0.283359


In [12]:
data

,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,vol_slope_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20
Date,,,,,,,,,,,,
2008-07-07,-1.220376,-1.882225,2.081321,-0.864592,-1.135796,-0.628471,-0.569023,-2.119483,1.416877,0.823997,-2.165333,-0.568676
2008-07-08,-1.211754,-1.882225,2.077736,-0.022097,-1.117078,-1.046961,-0.542321,-2.455523,1.450259,0.981827,-2.206775,-0.513460
2008-07-09,-0.957426,-1.882225,2.289506,1.636228,-1.218247,-0.338846,-0.561373,-1.988903,1.476862,0.785786,-1.960992,-0.565134
2008-07-10,-0.975703,-1.882225,2.287995,-0.006124,-1.013921,-0.474801,-0.596656,-2.401313,1.493754,0.505013,-1.940271,-0.562557
2008-07-11,-1.136144,-1.882225,2.391761,0.804554,-1.050675,-0.396400,-0.475448,-2.987661,1.507027,0.400347,-2.078101,-0.510752
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-15,0.375855,0.654743,-0.795604,0.021503,0.541634,-0.126723,-0.727148,-1.189892,-0.988672,-0.309058,0.314003,0.507783
2025-09-16,0.280399,0.654743,-0.833663,-0.287568,0.045456,-0.238463,-0.708679,-1.276291,-1.000565,-0.327333,0.453848,0.513354
2025-09-17,0.260114,0.654743,-0.836851,-0.019044,0.154682,-0.363408,-0.703390,-1.335486,-1.009414,-0.239281,0.522324,0.511624


In [13]:
close = all_data["Close"].copy()
close

Date
2008-07-07     4030.000000
2008-07-08     3988.550049
2008-07-09     4157.100098
2008-07-10     4162.200195
2008-07-11     4049.000000
                  ...     
2025-09-15    25069.199219
2025-09-16    25239.099609
2025-09-17    25330.250000
2025-09-18    25423.599609
2025-09-19    25327.050781
Name: Close, Length: 4219, dtype: float64

In [14]:
data.shape

(4219, 12)

In [15]:
# hyperparameters
long={'window': {'window_size': 180},
 'architecture': {'hidden_dim': 256, 'num_layers': 3},
 'contrastive': {'temperature': 0.06295392144086782, 'exclusion_radius': 19},
 'augmentation': {'mask_ratio': 0.4379409890039726,
  'jitter_sigma': 0.045869891350296864},
 'optimization': {'learning_rate': 0.00011624440419822255, 'batch_size': 64}}


short = {'window': {'window_size': 40},
 'architecture': {'hidden_dim': 128, 'num_layers': 3},
 'contrastive': {'temperature': 0.05865611450479419, 'exclusion_radius': 5},
 'augmentation': {'mask_ratio': 0.4,
  'jitter_sigma': 0.028089550039277756},
 'optimization': {'learning_rate': 0.00015072787810725861, 'batch_size': 32}}

mode="short"

if mode == "long":
    hyperparams = long
elif mode == "short":
    hyperparams = short
else:
    raise ValueError("Invalid mode. Choose 'long' or 'short'.")

In [16]:
class TimeSeriesWindowDataset(Dataset):
    def __init__(self, data, nifty_target, window_size=60):
        """
        Args:
            data: numpy array [T, D] containing features
            nifty_target: numpy array [T] containing 0 and 1 indicating 
                          whether Nifty is up or down the NEXT day.
            window_size: Lookback window length (e.g., 60 days)
        """
        self.data = torch.tensor(data, dtype=torch.float32)
        self.window_size = window_size
        
        # Ensure targets are typed appropriately for your loss function 
        # (LongTensor for CrossEntropyLoss, FloatTensor for BCEWithLogitsLoss)
        self.nifty_target = torch.tensor(nifty_target, dtype=torch.float32)

    def __len__(self):
        return len(self.data) - self.window_size + 1

    def __getitem__(self, idx):
        # 1. Extract feature window: Shape [window_size, D]
        x = self.data[idx : idx + self.window_size]
        
        # 2. Sequence/Multiple Prediction Target: Shape [window_size]
        # Maps perfectly 1:1 with each time step inside the 'x' window
        y_window = self.nifty_target[idx : idx + self.window_size]
        
        # 3. Single Prediction/Forecasting Target: Scalar shape []
        # Extracts the next-day target corresponding strictly to the very last step in 'x'
        y_last_day = self.nifty_target[idx + self.window_size - 1].unsqueeze(-1)
        
        # MUST return all components to your training loop!
        return x, y_window, y_last_day

window_size = hyperparams["window"]["window_size"]
val_size = 0.3

train_size = int(len(data) * (1 - val_size))
train_data = data.iloc[:train_size]
val_data = data.iloc[train_size:]
nifty_target = nifty_close["is_tomorrow_up"].values
nifty_target_train = nifty_target[:train_size]
nifty_target_val = nifty_target[train_size:]

train_dataset = TimeSeriesWindowDataset(train_data.values, nifty_target_train,   window_size)
val_dataset = TimeSeriesWindowDataset(val_data.values, nifty_target_val, window_size)
combined_dataset = TimeSeriesWindowDataset(data.values, nifty_target, window_size)

batch_size = hyperparams['optimization']['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# set batch size as max
combined_loader = DataLoader(combined_dataset, batch_size=len(combined_dataset), shuffle=False)



In [36]:
len(train_dataset) , len(val_dataset), len(combined_dataset)

(2914, 1227, 4180)

In [17]:
len(combined_dataset)

4180

In [18]:
for x, y_window, y_last_day in combined_loader:
    print("x shape:", x.shape)              # [batch_size, window_size, num_features]
    print("y_window shape:", y_window.shape)  # [batch_size, window_size]
    print("y_last_day shape:", y_last_day.shape)  # [batch_size]
    break

x shape: torch.Size([4180, 40, 12])
y_window shape: torch.Size([4180, 40])
y_last_day shape: torch.Size([4180, 1])


In [19]:
len(combined_dataset)

4180

In [20]:
for x, y_window, y_last_day in train_loader:
    print("x shape:", x.shape)              # [batch_size, window_size, num_features]
    print("y_window shape:", y_window.shape)  # [batch_size, window_size]
    print("y_last_day shape:", y_last_day.shape)  # [batch_size]
    print("y_window[0]:", y_window[3])  # First sequence of targets in the batch
    print("y_last_day[0]:", y_last_day[3])  # First single target in the batch
    break

x shape: torch.Size([32, 40, 12])
y_window shape: torch.Size([32, 40])
y_last_day shape: torch.Size([32, 1])
y_window[0]: tensor([0., 0., 0., 0., 1., 1., 1., 1., 1., 0., 0., 1., 0., 1., 1., 1., 0., 1.,
        1., 1., 1., 1., 0., 0., 0., 0., 0., 1., 0., 1., 1., 1., 0., 0., 1., 0.,
        1., 0., 0., 1.])
y_last_day[0]: tensor([1.])


In [21]:
nifty_target_train[:window_size]
# check if the first sequence of targets in the batch matches the first window_size elements of nifty_target_train using an assertion
# assert torch.equal(y_window[0], torch.tensor(nifty_target_train[:window_size], dtype=torch.float32)), "Mismatch between y_window[0] and nifty_target_train[:window_size]"

array([0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1,
       1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1])

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F



class AttentionFeatureFusion(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_heads=4):
        super().__init__()
        # Project each individual feature channel to a small embedding space
        self.feature_embed = nn.Linear(1, hidden_dim) 
        self.input_dim = input_dim
        self.dropout = nn.Dropout(p=0.1)
        
        # Cross-feature attention mechanism
        self.mha = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads, batch_first=True)
        
        # Compress the expanded feature dimension back to hidden_dim
        self.output_layer = nn.Linear(input_dim * hidden_dim, hidden_dim)
        
    def forward(self, x):
        # Input shape: [B, T, D] -> D is your number of features (Nifty, Gold, etc.)
        B, T, D = x.shape
        
        # 1. Isolate each feature and embed it: [B, T, D, 1] -> [B, T, D, H]
        x_expanded = x.unsqueeze(-1) 
        feat_embeddings = self.feature_embed(x_expanded) 
        
        # 2. Reshape to treat features as tokens for Multi-Head Attention: [B*T, D, H]
        feat_embeddings = feat_embeddings.view(B * T, D, -1)
        
        # 3. Compute dependencies between features at the exact same timestamp
        attn_out, _ = self.mha(feat_embeddings, feat_embeddings, feat_embeddings)
        
        # 4. Flatten features and project back to standard hidden_dim: [B, T, H]
        attn_out = attn_out.reshape(B, T, D * attn_out.size(-1))
        attn_out = self.dropout(attn_out)
        return self.output_layer(attn_out)

class CausalDilatedResidualBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()
        # Standard causal convolution uses padding = dilation * (kernel_size - 1) on the left
        self.padding = dilation * (3 - 1) 
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, dilation=dilation)
        self.relu = nn.ReLU()

    def forward(self, x):
        # x shape: [B, C, T]
        res = x
        
        # Block 1
        x = F.pad(x, (self.padding, 0)) # Pad left only
        x = self.relu(self.conv1(x))
        
        # Block 2
        x = F.pad(x, (self.padding, 0))
        x = self.relu(self.conv2(x))
        
        return x + res # Residual connection
    
class DynamicMarketLossWrapper(nn.Module):
    def __init__(self):
        super().__init__()
        # Initialize learnable log-variance weights (starts at 0, meaning weight = 1.0)
        self.log_var_t = nn.Parameter(torch.zeros(1))
        self.log_var_i = nn.Parameter(torch.zeros(1))

    def forward(self, loss_t, loss_i):
        # Compute dynamic coefficients based on homoscedastic uncertainty
        weight_t = torch.exp(-self.log_var_t)
        weight_i = torch.exp(-self.log_var_i)
        
        # Balanced loss formulation + regularization penalty terms
        balanced_loss = (weight_t * loss_t + self.log_var_t) + (weight_i * loss_i + self.log_var_i)
        
        return balanced_loss

class TS2VecModelCorrected(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=3, max_window=40):
        super().__init__()
        # 1. Feature Projection Layer
        # self.input_fc = nn.Linear(input_dim, hidden_dim)
        self.input_fc = AttentionFeatureFusion(input_dim, hidden_dim)
        
        # 2. Learnable Positional Embeddings
        self.pos_emb = nn.Parameter(torch.zeros(1, max_window, hidden_dim))
        
        # 3. Dilated Residual Network
        self.net = nn.Sequential(*[
            CausalDilatedResidualBlock(hidden_dim, dilation=2**i) 
            for i in range(num_layers)
        ])

    def forward(self, x):
        # x: [B, T, D]
        B, T, D = x.shape
        
        # Project and add positional structural details
        x = self.input_fc(x) # [B, T, H]
        x = x + self.pos_emb[:, :T, :]
        
        # Prepare for Conv1D
        x = x.permute(0, 2, 1) # [B, H, T]
        x = self.net(x)
        x = x.permute(0, 2, 1) # [B, T, H]
        
        # Normalize representations to the unit hypersphere
        return F.normalize(x, p=2, dim=-1)

In [23]:
data.shape

(4219, 12)

In [24]:

def ts2vec_contrastive_loss_vectorized(
    z1, z2,
    temperature=0.1,
    base_exclusion_radius=5
):
    """
    z1, z2: [B, T, C]
    """
    B, T, C = z1.shape
    device = z1.device

    # flatten (instance, time)
    z1_flat = z1.reshape(B*T, C)
    z2_flat = z2.reshape(B*T, C)

    # cosine similarity == dot product because embeddings are normalized
    sim = torch.matmul(z1_flat, z2_flat.T) / temperature   # [BT, BT]

    # ----- temporal negative mask -----
    exclusion_radius = min(base_exclusion_radius, (T - 1) // 2)

    if exclusion_radius == 0:
        return torch.tensor(0.0, device=device)

    # time index per row
    time_idx = torch.arange(T, device=device).repeat(B)     # [BT]

    # batch index per row
    batch_idx = torch.arange(B, device=device).repeat_interleave(T)

    # same batch & temporally close → mask out
    temporal_dist = torch.abs(time_idx[:, None] - time_idx[None, :])
    same_batch = batch_idx[:, None] == batch_idx[None, :]

    invalid_negatives = same_batch & (temporal_dist <= exclusion_radius)

    # allow diagonal (positive pairs)
    diag = torch.eye(B*T, device=device, dtype=torch.bool)
    invalid_negatives = invalid_negatives & (~diag)

    # mask invalid negatives
    sim = sim.masked_fill(invalid_negatives, -1e9)

    # positives are diagonal
    labels = torch.arange(B*T, device=device)

    return F.cross_entropy(sim, labels)


def financial_lead_lag_shifter(x, max_shift=3):
    """
    Randomly shifts individual feature channels along the time axis 
    to simulate market lead-lag delays.
    Input shape x: [B, T, D] (PyTorch Tensor)
    """
    x_np = x.cpu().numpy()
    B, T, D = x_np.shape
    x_augmented = x_np.copy()
    
    for b in range(B):
        for d in range(D):
            shift = np.random.randint(-max_shift, max_shift + 1)
            if shift == 0:
                continue
            elif shift > 0:
                # Feature leads (shifted forward, pad initial steps with edge value)
                x_augmented[b, shift:, d] = x_np[b, :-shift, d]
                x_augmented[b, :shift, d] = x_np[b, 0, d]
            else:
                # Feature lags (shifted backward, pad trailing steps with edge value)
                shift = abs(shift)
                x_augmented[b, :-shift, d] = x_np[b, shift:, d]
                x_augmented[b, -shift:, d] = x_np[b, -1, d]
                
    return torch.from_numpy(x_augmented).to(x.device)

def time_mask(x, mask_ratio=0.2):
    B, T, D = x.shape
    mask_len = int(T * mask_ratio)
    start = np.random.randint(0, T - mask_len)
    x = x.clone()
    x[:, start:start+mask_len, :] = 0
    return x


def jitter(x, sigma=0.02):
    return x + sigma * torch.randn_like(x)


def temporal_pooling(z):
    if z.size(1) % 2 == 1:
        z = z[:, :-1]
    return z.reshape(z.size(0), z.size(1)//2, 2, z.size(2)).mean(dim=2)



def compute_contrastive_loss(z1, z2, temperature=0.05, exclusion_radius=5):
    """
    Computes bidirectional instance and temporal contrastive loss.
    """
    B, T, C = z1.shape
    if T <= 1:
        return torch.tensor(0.0, device=z1.device)
        
    # --- TEMPORAL LOSS SETUP ---
    sim_t = torch.bmm(z1, z2.transpose(1, 2)) / temperature
    
    # 1. DYNAMIC BOUNDARY FIX: Cap the radius based on current pooled T
    effective_radius = min(exclusion_radius, T - 1)
    
    mask_t = torch.eye(T, dtype=torch.bool, device=z1.device)
    if effective_radius > 0:
        for r in range(1, effective_radius + 1):
            # Safe allocation now that effective_radius guarantees T - r > 0
            ones_tensor = torch.ones(T - r, device=z1.device)
            mask_t |= torch.eye(T, dtype=torch.bool, device=z1.device).diagonal_scatter(ones_tensor, offset=r)
            mask_t |= torch.eye(T, dtype=torch.bool, device=z1.device).diagonal_scatter(ones_tensor, offset=-r)
            
    diag_targets = torch.arange(T, device=z1.device).unsqueeze(0).expand(B, -1)
    
    invalid_mask = mask_t.unsqueeze(0).expand(B, -1, -1) & ~torch.eye(T, dtype=torch.bool, device=z1.device).unsqueeze(0)
    sim_t = sim_t.masked_fill(invalid_mask, -1e9)
    
    loss_t = F.cross_entropy(sim_t.transpose(1, 2), diag_targets) + F.cross_entropy(sim_t, diag_targets)
    
    # --- INSTANCE LOSS SETUP ---
    z1_i = z1.transpose(0, 1)
    z2_i = z2.transpose(0, 1)
    sim_i = torch.bmm(z1_i, z2_i.transpose(1, 2)) / temperature
    
    instance_targets = torch.arange(B, device=z1.device).unsqueeze(0).expand(T, -1)
    loss_i = F.cross_entropy(sim_i.transpose(1, 2), instance_targets) + F.cross_entropy(sim_i, instance_targets)
    
    # return (loss_t + loss_i) / 4.0
    return (loss_t / 2.0), (loss_i / 2.0)


def hierarchical_loss(z1, z2, loss_balancer, alpha=0.05, exclusion_radius=5):
    """
    Squeezes the time dimension hierarchically and tracks dynamically balanced loss.
    """
    total_loss = 0.0
    depth = 0
    
    while z1.size(1) >= 2:  
        # 1. Unpack the separate temporal and instance losses
        loss_t, loss_i = compute_contrastive_loss(
            z1, z2, 
            temperature=alpha, 
            exclusion_radius=exclusion_radius
        )
        
        # 2. Use the smart balancer to calculate the unified scale loss
        scale_loss = loss_balancer(loss_t, loss_i)
        
        # 3. Add the dynamically balanced scale loss to the total pool
        total_loss += scale_loss
        depth += 1
        
        # Max-pooling across time dimensions
        z1 = temporal_pooling(z1) 
        z2 = temporal_pooling(z2)
        
    return total_loss / max(depth, 1)

# create sample batch for testing
sample_batch = torch.randn(8, window_size, data.shape[1])  # [B, T, D]
z1 = torch.randn(1, 4, 2)  # [B, T, C]
z2 = torch.randn(1, 4, 2)  # [B, T, C]




In [25]:
# loss = hierarchical_loss(z1, z2, alpha=0.05, exclusion_radius=5)
# loss

In [26]:
data.shape

(4219, 12)

In [27]:
model = TS2VecModelCorrected(
    input_dim=data.shape[1], 
    hidden_dim=hyperparams['architecture']['hidden_dim'], 
    num_layers=hyperparams['architecture']['num_layers'],
    max_window=hyperparams['window']['window_size']
).to(device)

# optimizer = torch.optim.Adam(model.parameters(), lr=hyperparams['optimization']['learning_rate'])
# 1. Instantiate the module before the loop starts
loss_balancer = DynamicMarketLossWrapper().to(device)

# 2. Add its parameters to your optimizer so it can learn them
optimizer = torch.optim.Adam(
    list(model.parameters()) + list(loss_balancer.parameters()), 
    lr=hyperparams['optimization']['learning_rate'],
    weight_decay=1e-4  # Add this to constrain the network weights
)

In [28]:
model

TS2VecModelCorrected(
  (input_fc): AttentionFeatureFusion(
    (feature_embed): Linear(in_features=1, out_features=128, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (mha): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
    )
    (output_layer): Linear(in_features=1536, out_features=128, bias=True)
  )
  (net): Sequential(
    (0): CausalDilatedResidualBlock(
      (conv1): Conv1d(128, 128, kernel_size=(3,), stride=(1,))
      (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,))
      (relu): ReLU()
    )
    (1): CausalDilatedResidualBlock(
      (conv1): Conv1d(128, 128, kernel_size=(3,), stride=(1,), dilation=(2,))
      (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), dilation=(2,))
      (relu): ReLU()
    )
    (2): CausalDilatedResidualBlock(
      (conv1): Conv1d(128, 128, kernel_size=(3,), stride=(1,), dilation=(4,))
      (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), dil

In [29]:
%%time
num_epochs = 1

# probe_target_dim = 1 
# hidden_dim = hyperparams['architecture']['hidden_dim']
# linear_probe = nn.Linear(hidden_dim, probe_target_dim).to(device)
# probe_norm = nn.LayerNorm(hidden_dim).to(device)

# # A completely separate optimizer ensuring the probe doesn't affect TS2Vec weights
# probe_optimizer = torch.optim.Adam(linear_probe.parameters(), lr=0.005)
# # Pass both parameters to the probe optimizer
# probe_optimizer = torch.optim.Adam(
#     list(probe_norm.parameters()) + list(linear_probe.parameters()), 
#     lr=0.01  # Bumped to 0.01 to force it out of the 0.693 trough faster
# )
# probe_criterion = nn.BCEWithLogitsLoss()

for epoch in range(num_epochs):
    model.train()
    loss_balancer.train()  # Ensure the loss balancer is in training mode
    # linear_probe.train()  # Ensure the linear probe is in training mode
    train_loss = 0.0
    probe_train_loss = 0.0
    
    for x, _, y_last in train_loader:
        x = x.to(device)

        
        # Extract hyperparameter settings for readability
        mask_ratio = hyperparams['augmentation']['mask_ratio']
        jitter_sigma = hyperparams['augmentation']['jitter_sigma']
        max_shift = hyperparams['augmentation'].get('max_shift', 5) # safe fallback to 3
        
        # --- VIEW 1 GENERATION ---
        # 1. Apply structural lead-lag distortion
        x1 = financial_lead_lag_shifter(x, max_shift=max_shift)
        # 2. Simulate missing timeline windows
        x1 = time_mask(x1, mask_ratio=mask_ratio)
        # 3. Inject random signal noise
        x1 = jitter(x1, sigma=jitter_sigma)
        
        # --- VIEW 2 GENERATION ---
        # Apply the same pipeline sequentially (each step is stochastic, so views will be completely different)
        x2 = financial_lead_lag_shifter(x, max_shift=max_shift)
        x2 = time_mask(x2, mask_ratio=mask_ratio)
        x2 = jitter(x2, sigma=jitter_sigma)
        
        # Forward pass
        z1 = model(x1)
        z2 = model(x2)
        
        # Calculate Hierarchical Loss
        loss = hierarchical_loss(
            z1, z2, 
            loss_balancer=loss_balancer,
            alpha=hyperparams['contrastive']['temperature'], 
            exclusion_radius=hyperparams['contrastive']['exclusion_radius']
        )
        
        # Backward Pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        # # --- Online Linear Probe Training Phase ---
        # # 1. Get clean embeddings from current state of the model without tracking gradients for TS2Vec
        # with torch.no_grad():
        #     z_clean = model(x) # Shape: [B, T, H]
            
        # # 1. Take the last day step representation
        # z_last = z_clean[:, -1, :]  # Shape: [B, H]

        # # 2. Standardize the scale
        # z_normed = probe_norm(z_last)

        # # 3. Generate probe logit predictions
        # probe_preds = linear_probe(z_normed)  # Shape: [B, 1]

        # # 4. Enforce strict type matching for the targets
        # # y_last_day must be float, matching the exact shape of probe_preds [B, 1]
        # y_target = y_last.float().view(-1, 1)

        # probe_loss = probe_criterion(probe_preds, y_target)

        # probe_optimizer.zero_grad()
        # probe_loss.backward()
        # probe_optimizer.step()
        # probe_train_loss += probe_loss.item()
        
    # --- VALIDATION PHASE ---
    model.eval()
    loss_balancer.eval()  # Ensure the loss balancer is in eval mode
    # linear_probe.eval() # Put probe in evaluation mode
    val_loss = 0.0
    # probe_val_loss = 0.0
    train_embeddings, train_targets = [], []
    val_embeddings, val_targets = [], []
    
    with torch.no_grad():
        # Collect all training embeddings
        for x, _, y_last in train_loader:
            x = x.to(device)
            z = model(x)  # [B, T, H]
            # Isolate the last day step representation: [B, H]
            train_embeddings.append(z[:, -1, :].cpu().numpy())
            train_targets.append(y_last.cpu().numpy())

        for x, _, y_last in val_loader:
            x = x.to(device)

            

            
            z1 = model(x)
            z2 = model(x)
            z=z1  # For embedding collection, we can just use z1
            
            loss_val = hierarchical_loss(
                z1, z2, 
                loss_balancer=loss_balancer,
                alpha=hyperparams['contrastive']['temperature'], 
                exclusion_radius=hyperparams['contrastive']['exclusion_radius']
            )
            val_loss += loss_val.item()

            val_embeddings.append(z[:, -1, :].cpu().numpy())
            val_targets.append(y_last.cpu().numpy())

            # # --- Linear Probe Validation Phase ---
            # val_z_last = z1[:, -1, :]
            # val_preds = linear_probe(probe_norm(val_z_last))
            
            # y_val_target = y_last.float().view(-1, 1)
            # p_val_loss = probe_criterion(val_preds, y_val_target)
            # probe_val_loss += p_val_loss.item()


    # Concatenate lists into single clean NumPy arrays
    X_train_emb = np.concatenate(train_embeddings, axis=0)
    y_train_emb = np.concatenate(train_targets, axis=0).ravel() # flatten to 1D
    
    X_val_emb = np.concatenate(val_embeddings, axis=0)
    y_val_emb = np.concatenate(val_targets, axis=0).ravel()

    # --- 3. THE SCIKIT-LEARN LINEAR PROBE ---
    # We use a solid penalty (C=0.1 or 1.0) to keep it strictly regularized
    print(f"Length of training embeddings: {X_train_emb.shape}, Length of validation embeddings: {X_val_emb.shape}")
    probe = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    probe.fit(X_train_emb, y_train_emb)
    
    # Calculate Probabilities and Binary Predictions
    train_preds_proba = probe.predict_proba(X_train_emb)
    val_preds_proba = probe.predict_proba(X_val_emb)
    val_preds = probe.predict(X_val_emb)
    
    # Extract Scikit-Learn Metrics
    probe_train_loss = log_loss(y_train_emb, train_preds_proba)
    probe_val_loss = log_loss(y_val_emb, val_preds_proba)
    probe_val_acc = accuracy_score(y_val_emb, val_preds) * 100

    # Normalize metrics per epoch step bounds
    epoch_train_metric = train_loss / len(train_loader)
    epoch_val_metric = val_loss / len(val_loader)

        # epoch_probe_train = probe_train_loss / len(train_loader)
        # epoch_probe_val = probe_val_loss / len(val_loader)
    
    print(f"Epoch {epoch+1:02d}/{num_epochs} | Train Loss: {epoch_train_metric:.5f} | Val Loss: {epoch_val_metric:.5f} | Probe Train Loss: {probe_train_loss:.5f} | Probe Val Loss: {probe_val_loss:.5f} | Probe Val Acc: {probe_val_acc:.2f}%")

Length of training embeddings: (2914, 128), Length of validation embeddings: (1227, 128)
Epoch 01/1 | Train Loss: 4.09875 | Val Loss: 3.79517 | Probe Train Loss: 0.69122 | Probe Val Loss: 0.68925 | Probe Val Acc: 55.01%
CPU times: user 4.71 s, sys: 878 ms, total: 5.58 s
Wall time: 5.59 s


In [40]:
y_train_emb.shape

(2914,)

In [42]:
values, counts = np.unique(y_train_emb, return_counts=True)
values, counts

(array([0., 1.], dtype=float32), array([1389, 1525]))

In [32]:
2914+1227+hyperparams['window']['window_size']

4181

In [75]:
# save model
torch.save(model.state_dict(), f'./../../models/ts2vec_nifty_{mode}.pth')

In [ ]:
# load model
model.load_state_dict(torch.load(f'../../models/ts2vec_nifty_{mode}.pth'))

<All keys matched successfully>

In [76]:
all_embeddings = []
with torch.inference_mode():
    for x in combined_loader:
        # print("x.shape", x.shape)
        x = x.to(device)    
        z = model(x)          # [B, T, C]
        # z_mean = z.mean(dim=1)  # [B, C]
        z_mean = z[:, -1, :]  # use last time step embedding as representation
        all_embeddings.append(z_mean)

embeddings_full = torch.cat(all_embeddings).cpu().numpy()

In [77]:
len(all_embeddings)

1

In [78]:
embeddings_full.shape

(4180, 128)

In [79]:
val_df = pd.DataFrame(
    embeddings_full,
    index=data.index[window_size - 1:]
)

In [80]:
val_df["Close"] = all_data["Close"].iloc[window_size-1:].values

In [81]:
val_df.to_pickle('../../data/nifty_ts2vec_embeddings.pkl')

# read back
# with open('../../data/nifty_ts2vec_embeddings.pkl', 'rb') as f:
#     loaded_val_df = pickle.load(f)

In [30]:
with open('../../data/new_scaled_features.pkl', 'rb') as f:
    new_scaled_features = pickle.load(f)
new_df = new_scaled_features["scaled_features"]


In [31]:
new_df

,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,vol_slope_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20
Date,,,,,,,,,,,,
2025-06-27,0.798826,0.447462,-0.593711,0.009201,0.356613,-0.200888,-0.542173,0.544669,-0.628762,-0.596476,0.810645,-0.057347
2025-06-30,0.675988,0.447462,-0.582376,0.092789,0.115660,-0.301578,-0.531930,0.757795,-0.653870,-0.709449,0.701643,-0.100755
2025-07-01,0.981913,0.447462,-0.609857,-0.206111,0.231964,0.040351,-0.516026,0.603365,-0.677025,-0.652962,0.716140,-0.026861
2025-07-02,0.751418,0.447462,-0.608705,0.014383,0.236594,-0.217982,-0.565817,0.559314,-0.695986,-0.531682,0.636009,-0.000126
2025-07-03,0.526074,0.447462,-0.622690,-0.102189,0.137778,-0.693584,-0.583572,0.429694,-0.711442,-0.430338,0.595629,0.001498
...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-13,-0.479008,0.668348,-0.319403,0.578947,-0.187915,0.110005,0.076889,-1.308994,-0.812337,0.343861,-0.243405,0.225053
2026-02-16,-0.310087,0.668348,-0.290313,0.229518,-0.533300,0.318026,0.075401,-1.324313,-0.801075,0.342199,0.055013,0.135632
2026-02-17,-0.173470,0.668348,-0.296863,-0.044930,-0.746331,0.429565,0.057481,-0.837821,-0.796249,0.156126,0.110644,0.165542


In [ ]:
# convert to tensor of shape [T[idx : idx+Window], D]
new_dataset = TimeSeriesWindowDataset(new_df.values, window_size)
new_loader = DataLoader(new_dataset, batch_size=1, shuffle=False)
new_embeddings = []
model.eval()
loss_balancer.eval()
with torch.inference_mode():
    for x in new_loader:
        x = x.to(device)
        z = model(x)
        z_mean = z.mean(dim=1)
        # break
        new_embeddings.append(z_mean)
embeddings_ = torch.cat(new_embeddings).cpu().numpy()

In [68]:
embeddings_[0].shape

(128,)

2025-09-19 00:00:00

In [69]:
new_df = pd.DataFrame(embeddings_,
                      index=new_df.index[window_size-1:])
new_df

,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,127
Date,,,,,,,,,,,,,,,,,,,,,
2025-09-22,0.024366,0.016679,0.001446,0.0,0.000296,0.045045,0.042060,0.015169,0.018079,0.060219,...,0.046117,0.019970,0.000000,0.015816,0.118297,0.0,0.009491,0.000000,0.000896,0.054704
2025-09-23,0.032127,0.018201,0.006207,0.0,0.000193,0.036059,0.044192,0.015491,0.015427,0.065883,...,0.046117,0.022658,0.000000,0.013822,0.119071,0.0,0.008521,0.000000,0.000204,0.052041
2025-09-24,0.036056,0.017840,0.011655,0.0,0.000000,0.031033,0.047209,0.015292,0.013869,0.070268,...,0.046117,0.026075,0.000000,0.011653,0.119840,0.0,0.007423,0.000000,0.000000,0.046904
2025-09-25,0.036470,0.018000,0.017973,0.0,0.000386,0.029524,0.050833,0.014964,0.011014,0.071080,...,0.046117,0.028563,0.000000,0.010547,0.118944,0.0,0.005023,0.000000,0.000000,0.039292
2025-09-26,0.035231,0.018345,0.021695,0.0,0.000399,0.026336,0.053617,0.016982,0.008276,0.071641,...,0.046117,0.031045,0.000000,0.010178,0.117865,0.0,0.002377,0.000000,0.000000,0.033579
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-13,0.122133,0.000000,0.002236,0.0,0.065892,0.025297,0.037269,0.012507,0.000000,0.051424,...,0.002111,0.000000,0.017649,0.000000,0.000000,0.0,0.002284,0.000146,0.000000,0.217994
2026-02-16,0.120922,0.000000,0.000390,0.0,0.061576,0.020006,0.037269,0.015455,0.000000,0.052388,...,0.001855,0.000000,0.017775,0.000000,0.000000,0.0,0.001310,0.001001,0.000000,0.204451
2026-02-17,0.119993,0.000000,0.000000,0.0,0.058461,0.015654,0.038269,0.020194,0.000000,0.052054,...,0.001423,0.000000,0.016447,0.000000,0.000000,0.0,0.003356,0.000173,0.000000,0.193517


In [70]:
new_df["Close"]= new_scaled_features["Close"].iloc[window_size-1:]
new_df

,0,1,2,3,4,5,6,7,8,9,...,119,120,121,122,123,124,125,126,127,Close
Date,,,,,,,,,,,,,,,,,,,,,
2025-09-22,0.024366,0.016679,0.001446,0.0,0.000296,0.045045,0.042060,0.015169,0.018079,0.060219,...,0.019970,0.000000,0.015816,0.118297,0.0,0.009491,0.000000,0.000896,0.054704,25202.349609
2025-09-23,0.032127,0.018201,0.006207,0.0,0.000193,0.036059,0.044192,0.015491,0.015427,0.065883,...,0.022658,0.000000,0.013822,0.119071,0.0,0.008521,0.000000,0.000204,0.052041,25169.500000
2025-09-24,0.036056,0.017840,0.011655,0.0,0.000000,0.031033,0.047209,0.015292,0.013869,0.070268,...,0.026075,0.000000,0.011653,0.119840,0.0,0.007423,0.000000,0.000000,0.046904,25056.900391
2025-09-25,0.036470,0.018000,0.017973,0.0,0.000386,0.029524,0.050833,0.014964,0.011014,0.071080,...,0.028563,0.000000,0.010547,0.118944,0.0,0.005023,0.000000,0.000000,0.039292,24890.849609
2025-09-26,0.035231,0.018345,0.021695,0.0,0.000399,0.026336,0.053617,0.016982,0.008276,0.071641,...,0.031045,0.000000,0.010178,0.117865,0.0,0.002377,0.000000,0.000000,0.033579,24654.699219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-13,0.122133,0.000000,0.002236,0.0,0.065892,0.025297,0.037269,0.012507,0.000000,0.051424,...,0.000000,0.017649,0.000000,0.000000,0.0,0.002284,0.000146,0.000000,0.217994,25471.099609
2026-02-16,0.120922,0.000000,0.000390,0.0,0.061576,0.020006,0.037269,0.015455,0.000000,0.052388,...,0.000000,0.017775,0.000000,0.000000,0.0,0.001310,0.001001,0.000000,0.204451,25682.750000
2026-02-17,0.119993,0.000000,0.000000,0.0,0.058461,0.015654,0.038269,0.020194,0.000000,0.052054,...,0.000000,0.016447,0.000000,0.000000,0.0,0.003356,0.000173,0.000000,0.193517,25725.400391


In [71]:
# save new_df
with open('../../data/infer_ts2vec_embeddings.pkl', 'wb') as f:
    pickle.dump(new_df, f)

In [73]:
len(embeddings_full), len(data) # 4219

(4160, 4219)

In [74]:
data.tail(1).index # DatetimeIndex(['2025-09-19'], dtype='datetime64[ns]', name='Date', freq=None)

DatetimeIndex(['2025-09-19'], dtype='datetime64[ns]', name='Date', freq=None)

In [69]:
# full_df = data.iloc[window_size - 1 : ].copy()
# full_df.shape, embeddings_full.shape

In [77]:
val_df.tail(1).index # DatetimeIndex(['2025-09-18'], dtype='datetime64[ns]', name='Date', freq=None)

DatetimeIndex(['2025-09-19'], dtype='datetime64[ns]', name='Date', freq=None)

In [78]:
val_df.tail(1)

,0,1,2,3,4,5,6,7,8,9,...,119,120,121,122,123,124,125,126,127,Close
Date,,,,,,,,,,,,,,,,,,,,,
2025-09-19,0.014889,0.01376,0.0,0.0,0.000274,0.055722,0.040683,0.016491,0.021291,0.054494,...,0.018828,0.0,0.018891,0.117782,0.0,0.009961,0.0,0.002228,0.053978,25327.050781


In [13]:
loaded_val_df.tail()

,0,1,2,3,4,5,6,7,8,9,...,55,56,57,58,59,60,61,62,63,Close
Date,,,,,,,,,,,,,,,,,,,,,
2025-09-12,0.039898,0.0,0.0,0.0,0.037175,0.007709,0.0,0.198886,0.0,0.000691,...,0.021603,0.035282,0.060062,0.045333,0.006579,0.0,0.000000,0.128279,0.021880,25114.000000
2025-09-15,0.041106,0.0,0.0,0.0,0.040144,0.007308,0.0,0.199363,0.0,0.001671,...,0.021746,0.035920,0.062121,0.048278,0.007309,0.0,0.000000,0.124305,0.022726,25069.199219
2025-09-16,0.041802,0.0,0.0,0.0,0.042458,0.007110,0.0,0.198903,0.0,0.002719,...,0.024219,0.036828,0.063576,0.050335,0.006806,0.0,0.000000,0.120679,0.022671,25239.099609
2025-09-17,0.041898,0.0,0.0,0.0,0.044741,0.007148,0.0,0.198069,0.0,0.003467,...,0.026791,0.037300,0.064201,0.050577,0.006008,0.0,0.000584,0.117141,0.021781,25330.250000
2025-09-18,0.042400,0.0,0.0,0.0,0.048064,0.006878,0.0,0.196341,0.0,0.003795,...,0.028357,0.038166,0.064065,0.048919,0.007334,0.0,0.001005,0.112287,0.020587,25423.599609


In [ ]:
def train_model(hyperparams, model_name, num_epochs=9):
    """
    Train a TS2Vec model with given hyperparameters.
    
    Args:
        hyperparams: dict with 'window', 'architecture', 'contrastive', 
                     'augmentation', 'optimization' keys
        model_name: str, name for saving model (e.g., 'long', 'short')
        num_epochs: int, number of training epochs
    """
    # Extract window size (handle both nested dict and direct value)
    if isinstance(hyperparams['window'], dict):
        window_size = hyperparams['window']['window_size']
    else:
        window_size = hyperparams['window']
    
    val_size = 0
    train_size = int(len(data) * (1 - val_size))
    train_data = data.iloc[:train_size]
    val_data = data.iloc[train_size:]

    # Create datasets and loaders
    train_dataset = TimeSeriesWindowDataset(train_data.values, window_size)
    val_dataset = TimeSeriesWindowDataset(val_data.values, window_size)

    batch_size = hyperparams['optimization']['batch_size']
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model and optimizer
    model = TS2VecModel(
        input_dim=data.shape[1],
        hidden_dim=hyperparams['architecture']['hidden_dim'],
        num_layers=hyperparams['architecture']['num_layers']
    ).to(device)
    
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=hyperparams['optimization']['learning_rate']
    )

    # Training loop
    for epoch in range(num_epochs):
        total_loss = 0
        model.train()
        
        for x in train_loader:
            x = x.to(device)
            x1 = jitter(
                time_mask(x, mask_ratio=hyperparams['augmentation']['mask_ratio']),
                sigma=hyperparams['augmentation']['jitter_sigma']
            )
            x2 = jitter(
                time_mask(x, mask_ratio=hyperparams['augmentation']['mask_ratio']),
                sigma=hyperparams['augmentation']['jitter_sigma']
            )
            z1 = model(x1)
            z2 = model(x2)

            loss = hierarchical_ts2vec_loss_v2(
                z1, z2,
                temperature=hyperparams['contrastive']['temperature'],
                exclusion_radius=hyperparams['contrastive']['exclusion_radius']
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch}, Loss {total_loss / len(train_loader):.4f}")
    
    # Save model
    model_path = f'./../../models/ts2vec_nifty_{model_name}.pth'
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to {model_path}")


# Define hyperparameters
long_params = {
    'window': {'window_size': 180},
    'architecture': {'hidden_dim': 256, 'num_layers': 4},
    'contrastive': {'temperature': 0.10901392823581604, 'exclusion_radius': 18},
    'augmentation': {'mask_ratio': 0.10112014206194599, 'jitter_sigma': 0.019884546877356246},
    'optimization': {'learning_rate': 0.0012543308801674451, 'batch_size': 64}
}

short_params = {
    'window': 60,
    'architecture': {'hidden_dim': 128, 'num_layers': 4},
    'contrastive': {'temperature': 0.09418251159949012, 'exclusion_radius': 3},
    'augmentation': {'mask_ratio': 0.1153883102823196, 'jitter_sigma': 0.006324426296094188},
    'optimization': {'learning_rate': 0.0017009936483370917, 'batch_size': 32}
}

# Train both models
# train_model(long_params, 'long')
# train_model(short_params, 'short')